Gốc

In [2]:
import re
import urllib.parse
import requests
import asyncio
import aiohttp
from urllib.parse import urlparse
from bs4 import BeautifulSoup
import urllib3
import numpy as np
import warnings
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
TIMEOUT = 8
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    )
}

FEATURES_URL = [
    "IsHTTPS",
    "LetterRatioInURL",
    "NoOfSubDomain",
    "DegitRatioInURL",
    "SpacialCharRatioInURL",
    "DomainLength"
]

FEATURES_HTML = [
    "NoOfExternalRef",
    "NoOfSelfRef",
    "LineOfCode",
    "NoOfImage",
    "LargestLineLength",
    "HasDescription",
    "HasSocialNet",
    "NoOfJS",
    "URLTitleMatchScore",
]

SOCIAL_DOMAINS = {
    # Meta
    "facebook.com",
    "instagram.com",
    "threads.net",
    "messenger.com",

    # X / Twitter
    "twitter.com",
    "x.com",

    # LinkedIn
    "linkedin.com",

    # Video / streaming
    "youtube.com",
    "youtu.be",
    "twitch.tv",
    "vimeo.com",
    "bilibili.com",

    # Short video
    "tiktok.com",
    "douyin.com",

    # Community / forum
    "reddit.com",
    "quora.com",
    "medium.com",
    "discord.com",
    "discord.gg",

    # Messaging
    "telegram.org",
    "t.me",
    "whatsapp.com",
    "wa.me",
    "signal.org",
    "signal.me",
    "line.me",
    "wechat.com",
    "weixin.qq.com",
    "kakao.com",

    # Image / inspiration
    "pinterest.com",
    "flickr.com",
    "imgur.com",

    # Dev social
    "github.com",
    "gitlab.com",

    # Regional socials
    "weibo.com",
    "vk.com",
    "ok.ru",

    # Blogging / creator
    "substack.com",
    "patreon.com",
    "buymeacoffee.com",

    # Audio social
    "soundcloud.com",
    "spotify.com",

    # Others
    "snapchat.com",
    "tumblr.com",
    "mastodon.social"
}

def load_json(file_path: str) -> dict:
    import json
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            return json.load(file)
    except FileNotFoundError:
        print(f"No file directory")
        return {}
    except json.JSONDecodeError as e:
        print(f"No JSON file")
        return {}

def load_txt(file_path: str) -> set:
    import json
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            return {
                line.strip() for line in file
                if line.strip() and not line.strip().startswith('#')
            }
    except FileNotFoundError:
        print(f"No file directory")
        return set()
    except json.JSONDecodeError as e:
        print(f"No JSON file")
        return set()

def fetch_page(url):
    """
    Trả về (response, soup).
    Nếu lỗi trả về (None, None).
    """
    try:
        resp = requests.get(
            url,
            headers=HEADERS,
            timeout=TIMEOUT,
            allow_redirects=True,
            verify=False,
        )
        if resp.status_code >= 400:
            return resp, None
        soup = BeautifulSoup(resp.text, "html.parser")
        return resp, soup
    except Exception:
        return None, None

# def extract_features_url(url):
#     features = {feature: 0 for feature in FEATURES_URL}
#     if not url:
#         return features

#     try:
#         # Normalize: add scheme if missing
#         if not re.match(r'^https?://', url):
#             url = "http://" + url

#         parsed = urlparse(url)
#         host = parsed.hostname or ""
#         path = parsed.path or ""

#         # parsed = urlparse(url)
#         domain = parsed.netloc

#         # fetch page
#         resp, soup = fetch_page(url)

#         # IsHTTPS
#         features['IsHTTPS'] = 1 if (resp and resp.url.startswith("https://")) else 0

#         # --- DomainLength ---
#         features['DomainLength'] = len(domain)

#         # --- NoOfSubDomain ---
#         domain_parts = domain.split('.')
#         features['NoOfSubDomain'] = max(0, len(domain_parts) - 2)

#         # --- URL character counts (bỏ protocol, bỏ www., cắt ký tự cuối) ---
#         mod_str = url.replace('https://', '').replace('http://', '').replace('www.', '')
#         if len(mod_str) > 0:
#             mod_str = mod_str[:-1]

#         url_len = max(len(url) - 1, 1)

#         num_letters = sum(c.isalpha() for c in mod_str)
#         num_digits = sum(c.isdigit() for c in mod_str)

#         num_equals     = mod_str.count('=')
#         num_qmark      = mod_str.count('?')
#         num_ampersand  = mod_str.count('&')
#         total_special  = sum(not c.isalnum() for c in mod_str)
#         num_other_special = total_special - (num_equals + num_qmark + num_ampersand)

#         features['LetterRatioInURL']     = round(num_letters      / url_len, 3)
#         features['DegitRatioInURL']      = round(num_digits       / url_len, 3)
#         features['SpacialCharRatioInURL'] = round(num_other_special / url_len, 3)

#         return features

#     except Exception as e:
#         return {**features, "error": str(e)}

def extract_features_html(url):
    features = {feature: np.nan for feature in FEATURES_HTML}
    if not url:
        return features

    try:
        # Normalize: add scheme if missing
        if not re.match(r'^https?://', url):
            url = "http://" + url

        parsed = urlparse(url)
        base_host = parsed.hostname or ""

        # fetch page
        resp, soup = fetch_page(url)

        if resp is None or soup is None:
            return features

        html_text = resp.text
        lines = html_text.splitlines()

        features['LineOfCode'] = len(lines)
        features['LargestLineLength'] = max((len(line) for line in lines), default=0)
        features['NoOfImage'] = len(soup.find_all("img"))

        features['NoOfJS'] = len(
            [s for s in soup.find_all("script") if s.get("src")]
        ) + len(
            [s for s in soup.find_all("script") if not s.get("src") and s.string]
        )

        # --- NoOfExternalRef ---
        all_anchors = soup.find_all("a", href=True)
        cnt_external = 0
        for a in all_anchors:
            href = a["href"].strip()
            if href.startswith("http"):
                a_host = urlparse(href).hostname or ""
                if a_host and a_host != base_host:
                    cnt_external += 1
        features['NoOfExternalRef'] = cnt_external

        # --- NoOfSelfRef ---
        cnt_self = 0
        for a in all_anchors:
            href = a["href"].strip()
            if href.startswith("http"):
                a_host = urlparse(href).hostname or ""
                if a_host == base_host:
                    cnt_self += 1
            elif href.startswith("/") or not href.startswith(("mailto:", "tel:", "#", "javascript:")):
                cnt_self += 1
        features['NoOfSelfRef'] = cnt_self

        # --- HasDescription ---
        desc_tag = soup.find("meta", attrs={"name": re.compile(r"^description$", re.I)})
        if desc_tag and desc_tag.get("content", "").strip():
            features['HasDescription'] = 1
        else:
            features['HasDescription'] = 0

        # --- HasSocialNet ---
        has_social = 0
        for a in all_anchors:
            href = a["href"].strip()
            if href.startswith("http"):
                a_host = urlparse(href).hostname or ""
                a_host = re.sub(r'^www\.', '', a_host)
                if a_host in SOCIAL_DOMAINS:
                    has_social = 1
                    break
        features['HasSocialNet'] = has_social

        # --- URLTitleMatchScore ---
        # Công thức: LCS(domain_no_tld, title_lower) / len(domain_no_tld) * 100
        # domain_no_tld: bỏ www., bỏ TLD cuối cùng (ví dụ .com, .de, .uk)
        # LCS = Longest Common Substring (liên tiếp)
        title_tag = soup.find("title")
        title_text = title_tag.get_text().strip().lower() if title_tag else ""

        host_clean = re.sub(r'^www\.', '', base_host)
        domain_no_tld = host_clean.rsplit('.', 1)[0] if '.' in host_clean else host_clean

        if not domain_no_tld or not title_text:
            features['URLTitleMatchScore'] = 0.0
        else:
            lcs = lcs_length(domain_no_tld, title_text)
            features['URLTitleMatchScore'] = round(lcs / len(domain_no_tld) * 100, 6)

        return features

    except Exception as e:
        return {**features, "error": str(e)}

def lcs_length(s1, s2):
    """
    Trả về độ dài chuỗi con chung dài nhất (liên tiếp) giữa s1 và s2.
    Dùng để tính URLTitleMatchScore theo công thức dataset gốc:
        score = LCS(domain_no_tld, title_lower) / len(domain_no_tld) * 100
    """
    m, n = len(s1), len(s2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    best = 0
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i - 1] == s2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
                best = max(best, dp[i][j])
            else:
                dp[i][j] = 0
    return best

In [ ]:
import pandas as pd
import concurrent.futures
from tqdm import tqdm
import urllib3
import os
import time

# Tắt cảnh báo InsecureRequestWarning khi gọi verify=False
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

INPUT_FILE = "/content/drive/MyDrive/Đồ án tốt nghiệp/URL_legit_path.csv"
OUTPUT_FILE = "/content/drive/MyDrive/Đồ án tốt nghiệp/URL_legit_extracted_features.csv"

# [QUAN TRỌNG] Chỉnh giá trị này thành None nếu bạn muốn chạy toàn bộ 22.000 URLs!
# Hiện tại để 100 để test nhanh xem kết quả có đúng cấu trúc hay không
LIMIT_ROWS = 22167

MAX_WORKERS = 15 # Số luồng chạy song song

def process_single_url(row):
    """
    Trích xuất feature cho một URL độc lập
    """
    try:
        url = row['url']
        label = row.get('label', 'legit') # default legit nếu không có

        # Gọi hàm trích xuất
        url_feats = extract_features_url(url)
        html_feats = extract_features_html(url)

        # Kết hợp các feature vào 1 dict
        combined = {"url": url, "label": label}
        combined.update(url_feats)
        combined.update(html_feats)
        return combined
    except Exception as e:
        # Xử lý ngoại lệ trong trường hợp văng lỗi không mong muốn
        return {
            "url": row.get('url', ''),
            "label": row.get('label', 'legit'),
            "error": str(e)
        }

def main():
    print(f"Loading data from {INPUT_FILE}...")
    try:
        df = pd.read_csv(INPUT_FILE)
    except FileNotFoundError:
        print(f"Error: {INPUT_FILE} not found!")
        return

    print(f"Total rows in dataset: {len(df)}")

    if LIMIT_ROWS is not None:
        df = df.head(LIMIT_ROWS)
        print(f"[*] Limiting to first {LIMIT_ROWS} rows for testing.")
        print(f"[*] To process all URLs, open extract_batch_legit.py and set LIMIT_ROWS = None.")

    tasks = df.to_dict('records')
    results = []

    print(f"\nStarting extraction with {MAX_WORKERS} threads...\n")

    start_time = time.time()

    # Sử dụng multi-threading để gửi yêu cầu mạng đồng thời
    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(process_single_url, task): task for task in tasks}

        # Dùng tqdm để tạo thanh tiến trình
        for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures), desc="Extracting features"):
            res = future.result()
            results.append(res)

            # (Tuỳ chọn) Bạn có thể thêm tính năng auto-save sau mỗi 500 dòng ở đây
            # if len(results) % 500 == 0:
            #     pd.DataFrame(results).to_csv("checkpoint_" + OUTPUT_FILE, index=False)

    # Ghi dữ liệu ra file csv
    print("\nSaving final results...")
    out_df = pd.DataFrame(results)
    out_df.to_csv(OUTPUT_FILE, index=False)

    elapsed = time.time() - start_time
    print(f"Extraction complete! Saved {len(results)} rows to {OUTPUT_FILE}")
    print(f"Total time elapsed: {elapsed:.2f} seconds")

if __name__ == "__main__":
    main()


Loading data from /content/drive/MyDrive/Đồ án tốt nghiệp/URL_legit_path.csv...
Total rows in dataset: 22167
[*] Limiting to first 22167 rows for testing.
[*] To process all URLs, open extract_batch_legit.py and set LIMIT_ROWS = None.

Starting extraction with 15 threads...



Extracting features: 100%|██████████| 22167/22167 [00:00<00:00, 113462.91it/s]



Saving final results...
Extraction complete! Saved 22167 rows to /content/drive/MyDrive/Đồ án tốt nghiệp/URL_legit_extracted_features.csv
Total time elapsed: 2.97 seconds


Chậm/EXTRACT ALL

In [25]:
import os
import sys
import pandas as pd
import argparse
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

# # Thêm đường dẫn project vào sys.path để import src
# sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), '..')))

# from src.utils.feature_extraction import extract_features_url, extract_features_html, FEATURES_URL, FEATURES_HTML

def process_single_url(url, label):
    # Trích xuất đặc trưng URL
    url_features = extract_features_url(url)

    # Trích xuất đặc trưng HTML
    html_features = extract_features_html(url)

    # Kết hợp các đặc trưng
    combined = {"url": url}

    # URL features
    for f in FEATURES_URL:
        combined[f] = url_features.get(f, 0)

    # HTML features
    for f in FEATURES_HTML:
        combined[f] = html_features.get(f, -1)

    # Thông tin lỗi (nếu có)
    err_url = url_features.get("error")
    err_html = html_features.get("error")

    if err_url and err_html:
        combined["error"] = f"URL Error: {err_url} | HTML Error: {err_html}"
    elif err_url:
        combined["error"] = f"URL Error: {err_url}"
    elif err_html:
        combined["error"] = f"HTML Error: {err_html}"
    else:
        combined["error"] = None

    # Nhãn
    combined["label"] = label

    return combined

def main():
    parser = argparse.ArgumentParser(description="Extract features from URLs dataset for CatBoost model")
    parser.add_argument("--input", type=str, default="/content/drive/MyDrive/Đồ án tốt nghiệp/Datasets/urls.csv", help="Input CSV file with url and label")
    parser.add_argument("--output", type=str, default="/content/drive/MyDrive/Đồ án tốt nghiệp/Datasets/urls_extracted_clean.csv", help="Output CSV file with extracted features")
    parser.add_argument("--workers", type=int, default=64, help="Number of worker threads (để tăng tốc request mạng)")
    parser.add_argument("--limit", type=int, default=None, help="Limit number of rows to process (dùng để test)")
    parser.add_argument("--batch-size", type=int, default=500, help="Save to disk every N rows (lưu dự phòng tránh mất dữ liệu)")
    args, unknown = parser.parse_known_args()

    os.makedirs(os.path.dirname(args.output), exist_ok=True)

    print(f"Reading {args.input}...")
    try:
        df = pd.read_csv(args.input)
    except FileNotFoundError:
        print(f"Lỗi: Không tìm thấy file {args.input}")
        return

    if args.limit:
        df = df.head(args.limit)

    print(f"Tổng số dòng cần xử lý: {len(df)}")

    # Chuẩn bị danh sách
    items = df.to_dict('records')

    # Kiểm tra nếu file output đã tồn tại để resume (chạy tiếp)
    if os.path.exists(args.output):
        try:
            existing_df = pd.read_csv(args.output)
            processed_urls = set(existing_df['url'].values)
            items = [item for item in items if item['url'] not in processed_urls]
            print(f"Tìm thấy file {args.output} đã có {len(existing_df)} dòng.")
            print(f"Tiếp tục trích xuất cho {len(items)} dòng còn lại...")
        except Exception as e:
            print(f"Không thể đọc file output cũ: {e}")

    if not items:
        print("Đã hoàn tất xử lý tất cả dữ liệu.")
        return

    # Xử lý đa luồng với ThreadPoolExecutor
    new_results = []
    print(f"Bắt đầu trích xuất với {args.workers} luồng...")

    with ThreadPoolExecutor(max_workers=args.workers) as executor:
        future_to_item = {executor.submit(process_single_url, item['url'], item['label']): item for item in items}

        for i, future in enumerate(tqdm(as_completed(future_to_item), total=len(items))):
            try:
                res = future.result()
                new_results.append(res)
            except Exception as e:
                item = future_to_item[future]
                new_results.append({"url": item['url'], "label": item['label'], "error": str(e)})

            # Lưu theo lô nhỏ
            if (i + 1) % args.batch_size == 0:
                batch_df = pd.DataFrame(new_results)
                mode = 'a' if os.path.exists(args.output) else 'w'
                header = not os.path.exists(args.output)
                batch_df.to_csv(args.output, mode=mode, header=header, index=False)
                new_results = [] # Reset lại lô mới

    # Lưu nốt phần dư còn lại
    if new_results:
        batch_df = pd.DataFrame(new_results)
        mode = 'a' if os.path.exists(args.output) else 'w'
        header = not os.path.exists(args.output)
        batch_df.to_csv(args.output, mode=mode, header=header, index=False)

    print(f"\nHoàn tất! Kết quả được lưu tại {args.output}")

if __name__ == "__main__":
    main()

Reading /content/drive/MyDrive/Đồ án tốt nghiệp/Datasets/urls.csv...


/tmp/ipykernel_4263/3489045855.py:62: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(args.input)


Tổng số dòng cần xử lý: 320760
Tìm thấy file /content/drive/MyDrive/Đồ án tốt nghiệp/Datasets/urls_extracted_clean.csv đã có 209000 dòng.
Tiếp tục trích xuất cho 111758 dòng còn lại...
Bắt đầu trích xuất với 64 luồng...


  0%|          | 246/111758 [00:21<3:43:27,  8.32it/s]/tmp/ipykernel_4263/3830747800.py:157: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(resp.text, "html.parser")
 70%|██████▉   | 77827/111758 [7:45:20<7:22:13,  1.28it/s]WARNING:urllib3.connection:Failed to parse headers (url=https://aspnuke.it:44

KeyboardInterrupt: 

EXTRACT LEXICAL ONLY

In [7]:
import re
import math
from collections import Counter
from urllib.parse import urlparse, parse_qs


FEATURES_URL = [
    # existing
    "IsHTTPS",
    "LetterRatioInURL",
    "NoOfSubDomain",
    "DegitRatioInURL",
    "SpacialCharRatioInURL",
    "DomainLength",

    # length / structure
    "URLLength",
    "HostLength",
    "PathLength",
    "QueryLength",
    "PathDepth",
    "NumQueryParams",

    # suspicious structure
    "HasIPAddress",
    "HasAtSymbol",
    "HyphenCount",
    "DotCount",
    "SlashCount",
    "EqualCount",
    "QuestionMarkCount",
    "AmpersandCount",
    "PercentCount",

    # entropy
    "URLEntropy",
    "DomainEntropy",
    "PathEntropy",

    # keyword signals
    "SuspiciousKeywordCount",
    "SensitiveKeywordCount",
    "BrandKeywordCount",
]


SUSPICIOUS_KEYWORDS = [
    "login", "signin", "verify", "verification", "secure", "security",
    "account", "update", "confirm", "validate", "auth", "authentication",
    "wallet", "payment", "billing", "invoice", "recover", "unlock",
    "limited", "suspend", "alert", "warning", "support",
]

SENSITIVE_KEYWORDS = [
    "password", "passwd", "credential", "token", "otp", "2fa",
    "bank", "card", "credit", "debit", "ssn", "pin",
]

BRAND_KEYWORDS = [
    "paypal", "google", "microsoft", "apple", "facebook", "meta",
    "instagram", "amazon", "netflix", "steam", "binance",
    "bank", "chase", "wellsfargo", "dropbox", "onedrive",
]


def shannon_entropy(s: str) -> float:
    if not s:
        return 0.0

    counts = Counter(s)
    length = len(s)

    return -sum(
        (count / length) * math.log2(count / length)
        for count in counts.values()
    )


def has_ip_address(host: str) -> int:
    if not host:
        return 0

    ipv4_pattern = r"^\d{1,3}(\.\d{1,3}){3}$"
    return int(bool(re.match(ipv4_pattern, host)))


def count_keywords(text: str, keywords: list[str]) -> int:
    text = text.lower()
    return sum(1 for kw in keywords if kw in text)


def extract_features_url(url):
    features = {feature: 0 for feature in FEATURES_URL}

    if not url:
        return features

    try:
        raw_url = str(url).strip()

        if not re.match(r"^https?://", raw_url, re.I):
            normalized_url = "http://" + raw_url
        else:
            normalized_url = raw_url

        parsed = urlparse(normalized_url)

        host = parsed.hostname or ""
        path = parsed.path or ""
        query = parsed.query or ""

        netloc = parsed.netloc or ""
        full_without_scheme = re.sub(r"^https?://", "", normalized_url, flags=re.I)
        full_without_scheme = re.sub(r"^www\.", "", full_without_scheme, flags=re.I)

        url_len = max(len(full_without_scheme), 1)

        # =====================
        # Existing features
        # =====================
        features["IsHTTPS"] = int(normalized_url.lower().startswith("https://"))
        features["DomainLength"] = len(host)

        domain_parts = host.split(".") if host else []
        features["NoOfSubDomain"] = max(0, len(domain_parts) - 2)

        num_letters = sum(c.isalpha() for c in full_without_scheme)
        num_digits = sum(c.isdigit() for c in full_without_scheme)

        num_equals = full_without_scheme.count("=")
        num_qmark = full_without_scheme.count("?")
        num_ampersand = full_without_scheme.count("&")
        total_special = sum(not c.isalnum() for c in full_without_scheme)

        num_other_special = total_special - (
            num_equals + num_qmark + num_ampersand
        )

        features["LetterRatioInURL"] = round(num_letters / url_len, 6)
        features["DegitRatioInURL"] = round(num_digits / url_len, 6)
        features["SpacialCharRatioInURL"] = round(num_other_special / url_len, 6)

        # =====================
        # Length / structure
        # =====================
        features["URLLength"] = len(full_without_scheme)
        features["HostLength"] = len(host)
        features["PathLength"] = len(path)
        features["QueryLength"] = len(query)

        path_parts = [p for p in path.split("/") if p]
        features["PathDepth"] = len(path_parts)

        features["NumQueryParams"] = len(parse_qs(query))

        # =====================
        # Suspicious structure
        # =====================
        features["HasIPAddress"] = has_ip_address(host)
        features["HasAtSymbol"] = int("@" in full_without_scheme)

        features["HyphenCount"] = full_without_scheme.count("-")
        features["DotCount"] = full_without_scheme.count(".")
        features["SlashCount"] = full_without_scheme.count("/")
        features["EqualCount"] = full_without_scheme.count("=")
        features["QuestionMarkCount"] = full_without_scheme.count("?")
        features["AmpersandCount"] = full_without_scheme.count("&")
        features["PercentCount"] = full_without_scheme.count("%")

        # =====================
        # Entropy
        # =====================
        features["URLEntropy"] = round(shannon_entropy(full_without_scheme), 6)
        features["DomainEntropy"] = round(shannon_entropy(host), 6)
        features["PathEntropy"] = round(shannon_entropy(path), 6)

        # =====================
        # Keyword signals
        # =====================
        text = full_without_scheme.lower()

        features["SuspiciousKeywordCount"] = count_keywords(
            text,
            SUSPICIOUS_KEYWORDS,
        )

        features["SensitiveKeywordCount"] = count_keywords(
            text,
            SENSITIVE_KEYWORDS,
        )

        features["BrandKeywordCount"] = count_keywords(
            text,
            BRAND_KEYWORDS,
        )

        return features

    except Exception as e:
        return {**features, "error": str(e)}

In [8]:
import os
import re
import math
import argparse
import pandas as pd

from tqdm import tqdm
from urllib.parse import urlparse, parse_qs
from collections import Counter


FEATURES_URL = [
    # existing
    "IsHTTPS",
    "LetterRatioInURL",
    "NoOfSubDomain",
    "DegitRatioInURL",
    "SpacialCharRatioInURL",
    "DomainLength",

    # length / structure
    "URLLength",
    "HostLength",
    "PathLength",
    "QueryLength",
    "PathDepth",
    "NumQueryParams",

    # suspicious structure
    "HasIPAddress",
    "HasAtSymbol",
    "HyphenCount",
    "DotCount",
    "SlashCount",
    "EqualCount",
    "QuestionMarkCount",
    "AmpersandCount",
    "PercentCount",

    # entropy
    "URLEntropy",
    "DomainEntropy",
    "PathEntropy",

    # keyword signals
    "SuspiciousKeywordCount",
    "SensitiveKeywordCount",
    "BrandKeywordCount",
]


SUSPICIOUS_KEYWORDS = [
    "login", "signin", "verify", "verification", "secure", "security",
    "account", "update", "confirm", "validate", "auth", "authentication",
    "wallet", "payment", "billing", "invoice", "recover", "unlock",
    "limited", "suspend", "suspended", "alert", "warning", "support",
]

SENSITIVE_KEYWORDS = [
    "password", "passwd", "credential", "credentials", "token", "otp", "2fa",
    "bank", "card", "credit", "debit", "ssn", "pin",
]

BRAND_KEYWORDS = [
    "paypal", "google", "microsoft", "apple", "facebook", "meta",
    "instagram", "amazon", "netflix", "steam", "binance", "chase",
    "wellsfargo", "dropbox", "onedrive", "office", "outlook",
    "icloud", "dhl", "fedex", "ups",
]


def shannon_entropy(s: str) -> float:
    if not s:
        return 0.0

    counts = Counter(s)
    length = len(s)

    return -sum(
        (count / length) * math.log2(count / length)
        for count in counts.values()
    )


def has_ip_address(host: str) -> int:
    if not host:
        return 0

    ipv4_pattern = r"^\d{1,3}(\.\d{1,3}){3}$"

    if not re.match(ipv4_pattern, host):
        return 0

    parts = host.split(".")

    return int(all(0 <= int(part) <= 255 for part in parts))


def count_keywords(text: str, keywords: list[str]) -> int:
    text = text.lower()
    return sum(1 for kw in keywords if kw in text)


def extract_features_url(url):
    features = {feature: 0 for feature in FEATURES_URL}

    if not url:
        return features

    try:
        raw_url = str(url).strip()

        if not raw_url:
            return features

        if not re.match(r"^https?://", raw_url, re.I):
            normalized_url = "http://" + raw_url
        else:
            normalized_url = raw_url

        parsed = urlparse(normalized_url)

        host = parsed.hostname or ""
        path = parsed.path or ""
        query = parsed.query or ""

        full_without_scheme = re.sub(
            r"^https?://",
            "",
            normalized_url,
            flags=re.I,
        )

        full_without_scheme = re.sub(
            r"^www\.",
            "",
            full_without_scheme,
            flags=re.I,
        )

        url_len = max(len(full_without_scheme), 1)

        # =====================
        # Existing features
        # =====================
        features["IsHTTPS"] = int(normalized_url.lower().startswith("https://"))
        features["DomainLength"] = len(host)

        domain_parts = host.split(".") if host else []
        features["NoOfSubDomain"] = max(0, len(domain_parts) - 2)

        num_letters = sum(c.isalpha() for c in full_without_scheme)
        num_digits = sum(c.isdigit() for c in full_without_scheme)

        num_equals = full_without_scheme.count("=")
        num_qmark = full_without_scheme.count("?")
        num_ampersand = full_without_scheme.count("&")
        total_special = sum(not c.isalnum() for c in full_without_scheme)

        num_other_special = total_special - (
            num_equals + num_qmark + num_ampersand
        )

        features["LetterRatioInURL"] = round(num_letters / url_len, 6)
        features["DegitRatioInURL"] = round(num_digits / url_len, 6)
        features["SpacialCharRatioInURL"] = round(num_other_special / url_len, 6)

        # =====================
        # Length / structure
        # =====================
        features["URLLength"] = len(full_without_scheme)
        features["HostLength"] = len(host)
        features["PathLength"] = len(path)
        features["QueryLength"] = len(query)

        path_parts = [p for p in path.split("/") if p]
        features["PathDepth"] = len(path_parts)

        features["NumQueryParams"] = len(parse_qs(query, keep_blank_values=True))

        # =====================
        # Suspicious structure
        # =====================
        features["HasIPAddress"] = has_ip_address(host)
        features["HasAtSymbol"] = int("@" in full_without_scheme)

        features["HyphenCount"] = full_without_scheme.count("-")
        features["DotCount"] = full_without_scheme.count(".")
        features["SlashCount"] = full_without_scheme.count("/")
        features["EqualCount"] = full_without_scheme.count("=")
        features["QuestionMarkCount"] = full_without_scheme.count("?")
        features["AmpersandCount"] = full_without_scheme.count("&")
        features["PercentCount"] = full_without_scheme.count("%")

        # =====================
        # Entropy
        # =====================
        features["URLEntropy"] = round(shannon_entropy(full_without_scheme), 6)
        features["DomainEntropy"] = round(shannon_entropy(host), 6)
        features["PathEntropy"] = round(shannon_entropy(path), 6)

        # =====================
        # Keyword signals
        # =====================
        text = full_without_scheme.lower()

        features["SuspiciousKeywordCount"] = count_keywords(
            text,
            SUSPICIOUS_KEYWORDS,
        )

        features["SensitiveKeywordCount"] = count_keywords(
            text,
            SENSITIVE_KEYWORDS,
        )

        features["BrandKeywordCount"] = count_keywords(
            text,
            BRAND_KEYWORDS,
        )

        return features

    except Exception as e:
        return {**features, "error": str(e)}


def process_single_url(url, label):
    url_features = extract_features_url(url)

    row = {"url": url}

    for f in FEATURES_URL:
        row[f] = url_features.get(f, 0)

    row["error"] = url_features.get("error")
    row["label"] = label

    return row


def append_batch(results, output_path):
    if not results:
        return

    batch_df = pd.DataFrame(results)

    ordered_cols = ["url"] + FEATURES_URL + ["error", "label"]
    batch_df = batch_df.reindex(columns=ordered_cols)

    mode = "a" if os.path.exists(output_path) else "w"
    header = not os.path.exists(output_path)

    batch_df.to_csv(
        output_path,
        mode=mode,
        header=header,
        index=False,
    )


def main():
    parser = argparse.ArgumentParser(
        description="Extract lexical-only URL features for CatBoost model"
    )

    parser.add_argument(
        "--input",
        type=str,
        default="/content/drive/MyDrive/Đồ án tốt nghiệp/Datasets/urls.csv",
        help="Input CSV file with url and label",
    )

    parser.add_argument(
        "--output",
        type=str,
        default="/content/drive/MyDrive/Đồ án tốt nghiệp/Datasets/urls_lexical_extracted.csv",
        help="Output CSV file with lexical features only",
    )

    parser.add_argument(
        "--limit",
        type=int,
        default=None,
        help="Limit number of rows to process",
    )

    parser.add_argument(
        "--batch-size",
        type=int,
        default=5000,
        help="Save to disk every N rows",
    )

    parser.add_argument(
        "--force",
        action="store_true",
        help="Overwrite output file instead of resume",
    )

    args, unknown = parser.parse_known_args()

    output_dir = os.path.dirname(args.output)
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)

    if args.force and os.path.exists(args.output):
        os.remove(args.output)
        print(f"Đã xóa output cũ: {args.output}")

    print(f"Reading {args.input}...")

    try:
        df = pd.read_csv(args.input, low_memory=False)
    except FileNotFoundError:
        print(f"Lỗi: Không tìm thấy file {args.input}")
        return

    if "url" not in df.columns:
        raise ValueError("Input CSV phải có cột 'url'")

    if "label" not in df.columns:
        raise ValueError("Input CSV phải có cột 'label'")

    df = df[["url", "label"]].dropna(subset=["url", "label"]).copy()
    df["url"] = df["url"].astype(str)

    if args.limit:
        df = df.head(args.limit)

    print(f"Tổng số dòng cần xử lý: {len(df):,}")

    items = df.to_dict("records")

    # Resume
    if os.path.exists(args.output):
        try:
            existing_df = pd.read_csv(
                args.output,
                usecols=["url"],
                low_memory=False,
            )

            processed_urls = set(existing_df["url"].astype(str).values)

            items = [
                item for item in items
                if str(item["url"]) not in processed_urls
            ]

            print(f"Tìm thấy file output đã có {len(existing_df):,} dòng.")
            print(f"Tiếp tục trích xuất cho {len(items):,} dòng còn lại...")

        except Exception as e:
            print(f"Không thể đọc file output cũ để resume: {e}")
            print("Tiếp tục append, nên kiểm tra duplicate sau khi chạy xong.")

    if not items:
        print("Đã hoàn tất xử lý tất cả dữ liệu.")
        return

    results = []

    print("Bắt đầu trích xuất lexical features...")

    for i, item in enumerate(tqdm(items, total=len(items))):
        try:
            res = process_single_url(item["url"], item["label"])

        except Exception as e:
            res = {
                "url": item["url"],
                "label": item["label"],
                "error": str(e),
            }

            for f in FEATURES_URL:
                res[f] = 0

        results.append(res)

        if (i + 1) % args.batch_size == 0:
            append_batch(results, args.output)
            results = []

    append_batch(results, args.output)

    print(f"\nHoàn tất! Kết quả được lưu tại {args.output}")

    try:
        out_df = pd.read_csv(args.output, low_memory=False)
        print("\nOutput info:")
        print(out_df.info())

        print("\nLabel distribution:")
        print(out_df["label"].value_counts(dropna=False))

        print("\nMissing ratio:")
        print(out_df[FEATURES_URL].isna().mean().sort_values(ascending=False))

    except Exception as e:
        print(f"Không thể đọc lại output để kiểm tra: {e}")


if __name__ == "__main__":
    main()

Reading /content/drive/MyDrive/Đồ án tốt nghiệp/Datasets/urls.csv...
Tổng số dòng cần xử lý: 510,452
Bắt đầu trích xuất lexical features...


100%|██████████| 510452/510452 [01:09<00:00, 7380.91it/s]



Hoàn tất! Kết quả được lưu tại /content/drive/MyDrive/Đồ án tốt nghiệp/Datasets/urls_lexical_extracted.csv

Output info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 510452 entries, 0 to 510451
Data columns (total 30 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   url                     510452 non-null  object 
 1   IsHTTPS                 510452 non-null  int64  
 2   LetterRatioInURL        510452 non-null  float64
 3   NoOfSubDomain           510452 non-null  int64  
 4   DegitRatioInURL         510452 non-null  float64
 5   SpacialCharRatioInURL   510452 non-null  float64
 6   DomainLength            510452 non-null  int64  
 7   URLLength               510452 non-null  int64  
 8   HostLength              510452 non-null  int64  
 9   PathLength              510452 non-null  int64  
 10  QueryLength             510452 non-null  int64  
 11  PathDepth               510452 non-null  int64  
 12 

Mới


In [ ]:
import re
import asyncio
import aiohttp
import numpy as np
from urllib.parse import urlparse
from bs4 import BeautifulSoup

TIMEOUT = 8
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    )
}

FEATURES_URL = [
    "IsHTTPS", "LetterRatioInURL", "NoOfSubDomain",
    "DegitRatioInURL", "SpacialCharRatioInURL", "DomainLength"
]

FEATURES_HTML = [
    "NoOfExternalRef", "NoOfSelfRef", "LineOfCode", "NoOfImage",
    "LargestLineLength", "HasDescription", "HasSocialNet",
    "NoOfJS", "URLTitleMatchScore",
]

SOCIAL_DOMAINS = {
    "facebook.com","instagram.com","threads.net","messenger.com",
    "twitter.com","x.com","linkedin.com","youtube.com","youtu.be",
    "twitch.tv","vimeo.com","bilibili.com","tiktok.com","douyin.com",
    "reddit.com","quora.com","medium.com","discord.com","discord.gg",
    "telegram.org","t.me","whatsapp.com","wa.me","signal.org","signal.me",
    "line.me","wechat.com","weixin.qq.com","kakao.com","pinterest.com",
    "flickr.com","imgur.com","github.com","gitlab.com","weibo.com",
    "vk.com","ok.ru","substack.com","patreon.com","buymeacoffee.com",
    "soundcloud.com","spotify.com","snapchat.com","tumblr.com","mastodon.social"
}


async def fetch_page(session: aiohttp.ClientSession, url: str):
    """Fetch trang, trả về (final_url, html_text) hoặc (None, None) nếu lỗi."""
    try:
        async with session.get(url, allow_redirects=True) as resp:
            if resp.status >= 400:
                return str(resp.url), None
            text = await resp.text(errors="replace")
            return str(resp.url), text
    except Exception:
        return None, None


def extract_features_url(url: str, final_url: str | None) -> dict:
    features = {f: 0 for f in FEATURES_URL}
    try:
        parsed = urlparse(url)
        domain = parsed.netloc

        features["IsHTTPS"]      = 1 if (final_url and final_url.startswith("https://")) else 0
        features["DomainLength"] = len(domain)

        domain_parts = domain.split(".")
        features["NoOfSubDomain"] = max(0, len(domain_parts) - 2)

        mod_str = url.replace("https://", "").replace("http://", "").replace("www.", "")
        if mod_str:
            mod_str = mod_str[:-1]

        url_len = max(len(url) - 1, 1)
        num_letters       = sum(c.isalpha() for c in mod_str)
        num_digits        = sum(c.isdigit() for c in mod_str)
        total_special     = sum(not c.isalnum() for c in mod_str)
        num_punctuation   = mod_str.count("=") + mod_str.count("?") + mod_str.count("&")
        num_other_special = total_special - num_punctuation

        features["LetterRatioInURL"]      = round(num_letters       / url_len, 3)
        features["DegitRatioInURL"]       = round(num_digits        / url_len, 3)
        features["SpacialCharRatioInURL"] = round(num_other_special / url_len, 3)
    except Exception as e:
        features["error"] = str(e)
    return features


def extract_features_html(url: str, html_text: str | None) -> dict:
    features = {f: np.nan for f in FEATURES_HTML}
    if not html_text:
        return features
    try:
        parsed    = urlparse(url)
        base_host = parsed.hostname or ""
        soup      = BeautifulSoup(html_text, "html.parser")
        lines     = html_text.splitlines()

        features["LineOfCode"]        = len(lines)
        features["LargestLineLength"] = max((len(l) for l in lines), default=0)
        features["NoOfImage"]         = len(soup.find_all("img"))
        features["NoOfJS"]            = len([s for s in soup.find_all("script") if s.get("src")]) \
                                      + len([s for s in soup.find_all("script") if not s.get("src") and s.string])

        all_anchors  = soup.find_all("a", href=True)
        cnt_external = 0
        cnt_self     = 0
        has_social   = 0

        for a in all_anchors:
            href   = a["href"].strip()
            a_host = ""
            if href.startswith("http"):
                a_host = urlparse(href).hostname or ""

            # External
            if a_host and a_host != base_host:
                cnt_external += 1

            # Self
            if a_host == base_host:
                cnt_self += 1
            elif href.startswith("/") or not href.startswith(("mailto:", "tel:", "#", "javascript:")):
                cnt_self += 1

            # Social
            if not has_social and a_host:
                clean = re.sub(r"^www\.", "", a_host)
                if clean in SOCIAL_DOMAINS:
                    has_social = 1

        features["NoOfExternalRef"] = cnt_external
        features["NoOfSelfRef"]     = cnt_self
        features["HasSocialNet"]    = has_social

        desc_tag = soup.find("meta", attrs={"name": re.compile(r"^description$", re.I)})
        features["HasDescription"] = 1 if (desc_tag and desc_tag.get("content", "").strip()) else 0

        title_tag   = soup.find("title")
        title_text  = title_tag.get_text().strip().lower() if title_tag else ""
        host_clean  = re.sub(r"^www\.", "", base_host)
        domain_no_tld = host_clean.rsplit(".", 1)[0] if "." in host_clean else host_clean

        if domain_no_tld and title_text:
            lcs = lcs_length(domain_no_tld, title_text)
            features["URLTitleMatchScore"] = round(lcs / len(domain_no_tld) * 100, 6)
        else:
            features["URLTitleMatchScore"] = 0.0

    except Exception as e:
        features["error"] = str(e)
    return features


def lcs_length(s1: str, s2: str) -> int:
    m, n = len(s1), len(s2)
    dp   = [[0] * (n + 1) for _ in range(m + 1)]
    best = 0
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i-1] == s2[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
                best = max(best, dp[i][j])
            else:
                dp[i][j] = 0
    return best

In [ ]:
import os
import re
import asyncio
import aiohttp
import numpy as np
import pandas as pd
import argparse
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore", category=UserWarning)


# ── Cấu hình ──────────────────────────────────
CONCURRENT = 200        # số request đồng thời tối đa
TIMEOUT_S   = 8         # giây
BATCH_SIZE  = 500       # lưu file mỗi N dòng


# ── Helpers ───────────────────────────────────

def save_batch(results: list, output_path: str):
    df     = pd.DataFrame(results)
    mode   = "a" if os.path.exists(output_path) else "w"
    header = not os.path.exists(output_path)
    df.to_csv(output_path, mode=mode, header=header, index=False)


async def process_single_url(
    session:   aiohttp.ClientSession,
    semaphore: asyncio.Semaphore,
    url:       str,
    label:     int,
) -> dict:
    if not re.match(r"^https?://", url):
        url = "http://" + url

    async with semaphore:                          # giới hạn concurrent
        final_url, html_text = await fetch_page(session, url)

    url_features  = extract_features_url(url, final_url)
    html_features = extract_features_html(url, html_text)

    combined = {"url": url}
    for f in FEATURES_URL:
        combined[f] = url_features.get(f, 0)
    for f in FEATURES_HTML:
        combined[f] = html_features.get(f, np.nan)

    err_url  = url_features.get("error")
    err_html = html_features.get("error")
    if err_url and err_html:
        combined["error"] = f"URL: {err_url} | HTML: {err_html}"
    elif err_url:
        combined["error"] = f"URL: {err_url}"
    elif err_html:
        combined["error"] = f"HTML: {err_html}"
    else:
        combined["error"] = None

    combined["label"] = label
    return combined


async def run(args):
    # Đọc input
    print(f"Reading {args.input}...")
    try:
        df = pd.read_csv(args.input)
    except FileNotFoundError:
        print(f"Không tìm thấy file: {args.input}")
        return

    if args.limit:
        df = df.head(args.limit)

    items = df.to_dict("records")
    print(f"Tổng số dòng: {len(items)}")

    # Resume
    out_dir = os.path.dirname(args.output)
    if out_dir:
        os.makedirs(out_dir, exist_ok=True)

    if os.path.exists(args.output):
        try:
            existing       = pd.read_csv(args.output)
            processed_urls = set(existing["url"].values)
            items          = [r for r in items if r["url"] not in processed_urls]
            print(f"Resume: còn {len(items)} dòng chưa xử lý.")
        except Exception as e:
            print(f"Không đọc được file cũ: {e}")

    if not items:
        print("Đã xử lý hết.")
        return

    # Connector timeout
    timeout = aiohttp.ClientTimeout(total=TIMEOUT_S)
    connector = aiohttp.TCPConnector(
        limit=CONCURRENT,
        ssl=False,              # bỏ verify SSL như code gốc
        ttl_dns_cache=300,      # cache DNS 5 phút
        enable_cleanup_closed=True,
    )

    semaphore   = asyncio.Semaphore(CONCURRENT)
    new_results = []

    async with aiohttp.ClientSession(
        connector=connector,
        timeout=timeout,
        headers=HEADERS,
    ) as session:

        tasks = [
            process_single_url(session, semaphore, item["url"], item["label"])
            for item in items
        ]

        pbar = tqdm(total=len(tasks), desc="Extracting")
        for coro in asyncio.as_completed(tasks):
            try:
                result = await coro
            except Exception as e:
                result = {"url": "unknown", "label": -1, "error": str(e)}

            new_results.append(result)
            pbar.update(1)

            if len(new_results) % args.batch_size == 0:
                save_batch(new_results, args.output)
                new_results = []

        pbar.close()

    if new_results:
        save_batch(new_results, args.output)

    print(f"\nHoàn tất! Lưu tại: {args.output}")


# ── Entry point ───────────────────────────────

def get_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--input",      type=str, default="/content/drive/MyDrive/Đồ án tốt nghiệp/Datasets/urls_accessible.csv")
    parser.add_argument("--output",     type=str, default="/content/drive/MyDrive/Đồ án tốt nghiệp/Datasets/1000urls_accessible_extracted.csv")
    parser.add_argument("--limit",      type=int, default=5000)
    parser.add_argument("--batch-size", type=int, default=BATCH_SIZE)
    args, _ = parser.parse_known_args()
    return args

# Colab dùng asyncio.run() trực tiếp
args = get_args()
await run(args)          # Colab hỗ trợ top-level await

Reading /content/drive/MyDrive/Đồ án tốt nghiệp/Datasets/urls_accessible.csv...
Tổng số dòng: 5000


Extracting: 100%|██████████| 5000/5000 [03:50<00:00, 21.66it/s]


Hoàn tất! Lưu tại: /content/drive/MyDrive/Đồ án tốt nghiệp/Datasets/1000urls_accessible_extracted.csv


In [ ]:
import re
import requests
import pandas as pd
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    )
}

def check_url_fast(row):
    url = row['url']
    label = row['label']
    if not re.match(r'^https?://', url):
        url = "http://" + url
    try:
        resp = requests.head(url, headers=HEADERS, timeout=5, allow_redirects=True, verify=False)
        if resp.status_code == 405:
            resp = requests.get(url, headers=HEADERS, timeout=5, allow_redirects=True, verify=False, stream=True)
        accessible = resp.status_code < 400
        return {"url": row['url'], "label": label, "accessible": accessible, "status_code": resp.status_code}
    except Exception:
        return {"url": row['url'], "label": label, "accessible": False, "status_code": None}

df = pd.read_csv("/content/drive/MyDrive/Đồ án tốt nghiệp/Datasets/urls.csv")

print(f"Tổng số URL: {len(df)}")
print(f"Phân bố label:\n{df['label'].value_counts()}\n")

results = []
with ThreadPoolExecutor(max_workers=64) as executor:
    futures = {executor.submit(check_url_fast, row): row for _, row in df.iterrows()}
    for future in tqdm(as_completed(futures), total=len(df), desc="Checking URLs"):
        results.append(future.result())

result_df = pd.DataFrame(results)

# Tổng hợp
total = len(result_df)
inaccessible = result_df[~result_df['accessible']]
accessible = result_df[result_df['accessible']]

print(f"\n{'='*40}")
print(f"Tổng URL:               {total}")
print(f"Truy cập được:          {len(accessible)} ({len(accessible)/total*100:.1f}%)")
print(f"Không truy cập được:    {len(inaccessible)} ({len(inaccessible)/total*100:.1f}%)")
print(f"{'='*40}")
print(f"\nTheo label:")
summary = result_df.groupby('label')['accessible'].value_counts().rename("count")
print(summary.to_string())

print(f"\nStatus code phổ biến của URL lỗi:")
print(result_df[~result_df['accessible']]['status_code'].value_counts(dropna=False).head(10).to_string())

accessible_df = result_df[result_df['accessible']]
accessible_df.to_csv(
    "/content/drive/MyDrive/Đồ án tốt nghiệp/Datasets/urls_accessible.csv",
    index=False
)

inaccessible_df = result_df[~result_df['accessible']]
inaccessible_df.to_csv(
    "/content/drive/MyDrive/Đồ án tốt nghiệp/Datasets/urls_inaccessible.csv",
    index=False
)

print(f"Đã lưu {len(accessible_df)} URL accessible")

Tổng số URL: 500512
Phân bố label:
label
1    250612
0    249900
Name: count, dtype: int64



Checking URLs:  92%|█████████▏| 462282/500512 [1:26:32<09:45, 65.32it/s]WARNING:urllib3.connection:Failed to parse headers (url=https://aspnuke.it:443/): [MissingHeaderBodySeparatorDefect()], unparsed data: 'Spiders noindex: X-Robots-Tag: noindex\r\nDate: Sun, 10 May 2026 06:42:54 GMT\r\n\r\n'
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connection.py", line 568, in getresponse
    assert_header_parsing(httplib_response.msg)
  File "/usr/local/lib/python3.12/dist-packages/urllib3/util/response.py", line 88, in assert_header_parsing
    raise HeaderParsingError(defects=defects, unparsed_data=unparsed_data)
urllib3.exceptions.HeaderParsingError: [MissingHeaderBodySeparatorDefect()], unparsed data: 'Spiders noindex: X-Robots-Tag: noindex\r\nDate: Sun, 10 May 2026 06:42:54 GMT\r\n\r\n'
Checking URLs:  92%|█████████▏| 462297/500512 [1:26:32<09:48, 64.92it/s]WARNING:urllib3.connection:Failed to parse headers (url=https://aspnuke.it:443/login.php)


Tổng URL:               500512
Truy cập được:          275719 (55.1%)
Không truy cập được:    224793 (44.9%)

Theo label:
label  accessible
0      False         197144
       True           52756
1      True          222963
       False          27649

Status code phổ biến của URL lỗi:
status_code
502.0    118802
403.0     29529
NaN       28300
429.0     21809
404.0     19069
521.0      2404
500.0      2358
503.0       498
400.0       478
406.0       344
Đã lưu 275719 URL accessible


In [14]:
import pandas as pd
from urllib.parse import urlparse

# Đọc file CSV
df = pd.read_csv("/content/drive/MyDrive/Đồ án tốt nghiệp/Datasets/urls.csv")

# Giả sử file có cột 'url' và cột 'label'
def has_path(u):
    parsed = urlparse(u)
    return bool(parsed.path and parsed.path != "/")

# Thêm cột đánh dấu có path hay không
df['has_path'] = df['url'].apply(has_path)

# Đếm theo từng label
counts = df.groupby('label')['has_path'].sum()        # số lượng có path
totals = df.groupby('label')['has_path'].count()      # tổng số URL mỗi label
ratios = counts / totals                              # tỉ lệ

print("Số lượng có path theo label:")
print(counts)
print("\nTổng số URL theo label:")
print(totals)
print("\nTỉ lệ có path theo label:")
print(ratios)


/tmp/ipykernel_523/3469899745.py:5: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/content/drive/MyDrive/Đồ án tốt nghiệp/Datasets/urls.csv")


Số lượng có path theo label:
label
0     43389
1    204302
Name: has_path, dtype: int64

Tổng số URL theo label:
label
0     97797
1    222963
Name: has_path, dtype: int64

Tỉ lệ có path theo label:
label
0    0.443664
1    0.916304
Name: has_path, dtype: float64


In [29]:
import pandas as pd

# Đọc file CSV
# df = pd.read_csv("/content/drive/MyDrive/Đồ án tốt nghiệp/Datasets/1000urls_accessible_extracted.csv")
df = pd.read_csv("/content/drive/MyDrive/Đồ án tốt nghiệp/Datasets/phishstats_ml_dataset.csv")

# Đếm số lượng NaN theo từng label
# nan_counts = df.groupby("label")["LargestLineLength"].apply(lambda x: x.isna().sum())
# print(nan_counts)
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 259840 entries, 0 to 259839
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   url     259840 non-null  object
 1   label   259840 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 4.0+ MB


In [1]:
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
from bs4 import BeautifulSoup
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

BLOCKED_KEYWORDS = [
    "you have been blocked",
    "access denied",
    "attention required",
    "enable javascript and cookies",
    "ddos protection",
    "ray id",
    "sorry, you have been",
    "captcha",
    "please verify you are a human",
]

DEAD_STATUS = {502, 503, 504, 521, 522, 523, 524, 530}

MIN_BODY_SIZE = 500   # bytes — trang quá nhỏ thì không extract được gì
MAX_BODY_SIZE = 5_000_000  # 5MB — tránh download file lớn


def check_url_html(url, timeout=7):
    """
    Trả về dict:
    {
        "status": "valid" | "blocked" | "dead" | "no_html",
        "status_code": int,
        "reason": str,
        "html_size": int,          # bytes
        "num_tags": int,           # số lượng tag HTML
        "has_images": bool,
        "has_links": bool,
        "has_scripts": bool,
    }
    """
    result = {
        "status": "dead",
        "status_code": None,
        "reason": "",
        "html_size": 0,
        "num_tags": 0,
        "has_images": False,
        "has_links": False,
        "has_scripts": False,
    }

    try:
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                          "AppleWebKit/537.36 (KHTML, like Gecko) "
                          "Chrome/120.0.0.0 Safari/537.36",
            "Accept": "text/html,application/xhtml+xml,*/*",
            "Accept-Language": "en-US,en;q=0.9",
        }

        res = requests.get(
            url,
            headers=headers,
            timeout=timeout,
            allow_redirects=True,
            verify=False,
            stream=True,
        )

        result["status_code"] = res.status_code

        # Server lỗi hẳn
        if res.status_code in DEAD_STATUS:
            result["reason"] = f"server_error_{res.status_code}"
            return result

        # Kiểm tra Content-Type có phải HTML không
        content_type = res.headers.get("Content-Type", "").lower()
        if "text/html" not in content_type and "text/plain" not in content_type:
            result["status"] = "no_html"
            result["reason"] = f"content_type: {content_type}"
            return result

        # Đọc body với giới hạn kích thước
        raw_bytes = b""
        for chunk in res.iter_content(chunk_size=8192):
            raw_bytes += chunk
            if len(raw_bytes) >= MAX_BODY_SIZE:
                break

        html_size = len(raw_bytes)
        result["html_size"] = html_size

        # Body quá nhỏ → trang lỗi / rỗng
        if html_size < MIN_BODY_SIZE:
            result["status"] = "no_html"
            result["reason"] = f"body_too_small: {html_size} bytes"
            return result

        # Decode
        html_text = raw_bytes.decode("utf-8", errors="ignore").lower()

        # Bị chặn bot
        if any(kw in html_text for kw in BLOCKED_KEYWORDS):
            result["status"] = "blocked"
            result["reason"] = "bot_blocked"
            return result

        # Parse HTML sơ bộ để xác nhận có thể extract features
        soup = BeautifulSoup(raw_bytes, "html.parser")

        num_tags    = len(soup.find_all())
        has_images  = len(soup.find_all("img")) > 0
        has_links   = len(soup.find_all("a")) > 0
        has_scripts = len(soup.find_all("script")) > 0

        # Quá ít tag → không phải trang thật
        if num_tags < 5:
            result["status"] = "no_html"
            result["reason"] = f"too_few_tags: {num_tags}"
            return result

        result.update({
            "status": "valid",
            "reason": "ok",
            "num_tags": num_tags,
            "has_images": has_images,
            "has_links": has_links,
            "has_scripts": has_scripts,
        })
        return result

    except requests.exceptions.Timeout:
        result["reason"] = "timeout"
    except requests.exceptions.ConnectionError:
        result["reason"] = "connection_error"
    except requests.exceptions.SSLError:
        result["reason"] = "ssl_error"
    except Exception as e:
        result["reason"] = str(e)[:80]

    return result


def filter_valid_html_urls(input_csv, output_csv, max_workers=30, timeout=7):
    df = pd.read_csv(input_csv)
    df.drop_duplicates(subset=["url"], inplace=True)
    urls = df["url"].tolist()

    print(f"🔍 Kiểm tra {len(urls):,} URL (HTML parseable)...")

    all_results = {}

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_url = {
            executor.submit(check_url_html, url, timeout): url
            for url in urls
        }
        for future in tqdm(as_completed(future_to_url), total=len(urls)):
            url = future_to_url[future]
            try:
                all_results[url] = future.result()
            except Exception as e:
                all_results[url] = {"status": "dead", "reason": str(e)}

    # Gắn kết quả vào dataframe
    df["status"]      = df["url"].map(lambda u: all_results[u].get("status"))
    df["reason"]      = df["url"].map(lambda u: all_results[u].get("reason"))
    df["html_size"]   = df["url"].map(lambda u: all_results[u].get("html_size", 0))
    df["num_tags"]    = df["url"].map(lambda u: all_results[u].get("num_tags", 0))
    df["has_images"]  = df["url"].map(lambda u: all_results[u].get("has_images", False))
    df["has_links"]   = df["url"].map(lambda u: all_results[u].get("has_links", False))
    df["has_scripts"] = df["url"].map(lambda u: all_results[u].get("has_scripts", False))

    # Tách nhóm
    valid_df   = df[df["status"] == "valid"]
    blocked_df = df[df["status"] == "blocked"]
    no_html_df = df[df["status"] == "no_html"]
    dead_df    = df[df["status"] == "dead"]

    # Lưu file — chỉ giữ cột cần thiết cho ML
    ml_cols = ["url", "label", "html_size", "num_tags",
               "has_images", "has_links", "has_scripts"]

    existing_cols = [c for c in ml_cols if c in valid_df.columns]
    valid_df[['url', 'label']].to_csv(output_csv, index=False, encoding="utf-8")

    blocked_df.to_csv("blocked_urls.csv", index=False)
    dead_df.to_csv("dead_urls.csv",       index=False)
    no_html_df.to_csv("no_html_urls.csv", index=False)

    total = len(df)
    print(f"\n✅ Valid HTML  : {len(valid_df):,}   ({len(valid_df)/total*100:.1f}%)")
    print(f"🚧 Bị chặn    : {len(blocked_df):,}  ({len(blocked_df)/total*100:.1f}%)")
    print(f"📄 Không HTML  : {len(no_html_df):,}  ({len(no_html_df)/total*100:.1f}%)")
    print(f"💀 Dead        : {len(dead_df):,}  ({len(dead_df)/total*100:.1f}%)")

    # Thống kê lý do chết
    print("\n📋 Top lý do không dùng được:")
    reason_counts = df[df["status"] != "valid"]["reason"].value_counts().head(10)
    print(reason_counts.to_string())

    return valid_df


if __name__ == "__main__":
    filter_valid_html_urls(
        input_csv="/content/drive/MyDrive/Đồ án tốt nghiệp/Datasets/phishstats_ml_dataset.csv",
        output_csv="phishing_valid_html.csv",
        max_workers=30,  # Giảm xuống vì giờ đọc full body, nặng hơn
        timeout=7
    )

🔍 Kiểm tra 259,840 URL (HTML parseable)...


100%|██████████| 259840/259840 [1:42:59<00:00, 42.05it/s]



✅ Valid HTML  : 97,797   (37.6%)
🚧 Bị chặn    : 39,836  (15.3%)
📄 Không HTML  : 21,227  (8.2%)
💀 Dead        : 100,980  (38.9%)

📋 Top lý do không dùng được:
reason
connection_error             84268
bot_blocked                  39836
timeout                      11598
content_type:                 3448
server_error_503              3018
body_too_small: 355 bytes     2313
body_too_small: 142 bytes     1822
body_too_small: 0 bytes       1595
server_error_502              1241
body_too_small: 35 bytes       933


Extract MỚI NHẤT